# 03_ml_training: Train and Compare Models
Train 2+ Spark ML models on the 2012–2015 loans, tune them, and track every run in MLflow (REQ-03).

In [0]:
%run ./00_config

## Step 1: Load train, test and the Pipeline recipe

In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline

train = spark.table(TRAIN_TABLE)
test  = spark.table(TEST_TABLE)
feature_pipeline = Pipeline.load(PIPELINE_PATH)   # the 5-stage recipe saved in 02

print(f"Train: {train.count():,} | Test: {test.count():,} | Pipeline stages: {len(feature_pipeline.getStages())}")

## Step 2: Imbalance strategy: class weights
Each default gets a weight of (repaid count ÷ default count), so both classes count equally in training.
Chosen over oversampling (duplicates → overfitting) and undersampling (throws away real data).

In [0]:
# Count each class in TRAIN
n_repaid  = train.filter(F.col("label") == 0).count()
n_default = train.filter(F.col("label") == 1).count()
default_weight = n_repaid / n_default

# Add the weight column: defaults get the bigger weight, repaid loans get 1.0
train = train.withColumn(
    "class_weight",
    F.when(F.col("label") == 1, default_weight).otherwise(1.0)
)

print(f"Repaid: {n_repaid:,} | Defaults: {n_default:,} | Default weight: {default_weight:.2f}")

Train: 434,107 repaid vs 113,714 defaults → each default gets weight **3.82**, repaid loans get 1.0.

## Step 3: Set the MLflow experiment (all runs are recorded here)

In [0]:
import mlflow

mlflow.set_experiment(EXPERIMENT_PATH)
print("MLflow experiment:", EXPERIMENT_PATH)

MLflow experiment created: `loan_default_experiments` (in my workspace folder).

## Step 4: Validation split for tuning (fit 2012–2014, validate 2015)
Settings are compared on 2015, so the 2016 test set stays untouched until final evaluation.

In [0]:
# Practice-exam split inside TRAIN, also by time
fit_df = train.filter(F.col("issue_year") <= 2014)   # model learns here
val_df = train.filter(F.col("issue_year") == 2015)   # settings are compared here

print(f"Fit (2012-2014): {fit_df.count():,} | Validation (2015): {val_df.count():,}")

Fit set: 362,148 loans (2012–2014). Validation set: 185,673 loans (2015).

## Step 5: Scoring helper (AUC-ROC, AUC-PR, precision, recall, F1 for the default class)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

def get_metrics(preds):
    """Scores for predictions that have 'label', 'prediction' and 'rawPrediction' columns."""
    auc_roc = BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(preds)
    auc_pr  = BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(preds)

    # Count true positives, false positives, false negatives (positive = default)
    c = preds.agg(
        F.sum(((F.col("prediction") == 1) & (F.col("label") == 1)).cast("int")).alias("tp"),
        F.sum(((F.col("prediction") == 1) & (F.col("label") == 0)).cast("int")).alias("fp"),
        F.sum(((F.col("prediction") == 0) & (F.col("label") == 1)).cast("int")).alias("fn"),
    ).first()

    precision = c.tp / (c.tp + c.fp) if (c.tp + c.fp) else 0.0
    recall    = c.tp / (c.tp + c.fn) if (c.tp + c.fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"auc_roc": auc_roc, "auc_pr": auc_pr, "precision": precision, "recall": recall, "f1": f1}

print("get_metrics() ready")

## Step 6: Run 1: Logistic Regression (default settings), tracked in MLflow

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label",
                        weightCol="class_weight", maxIter=50, regParam=0.0)
model_pipeline = Pipeline(stages=feature_pipeline.getStages() + [lr])

with mlflow.start_run(run_name="logreg_default"):
    fitted = model_pipeline.fit(fit_df)                     # learn on 2012-2014
    metrics = get_metrics(fitted.transform(val_df))         # score on 2015

    mlflow.log_params({"algorithm": "LogisticRegression", "regParam": 0.0,
                       "maxIter": 50, "imbalance": "class_weight"})
    mlflow.log_metrics({f"val_{k}": v for k, v in metrics.items()})

print({k: round(v, 4) for k, v in metrics.items()})

Run 1, Logistic Regression (default): validation AUC-ROC **0.710**, recall **58.7%**, precision **41.8%**, F1 0.488.

## Step 7: Training helper + tuning Logistic Regression (regParam = 0.01, 0.1)

In [0]:
def train_and_log(run_name, classifier, params):
    """Build pipeline (5 stages + model), learn on 2012-2014, score on 2015, record in MLflow."""
    pipe = Pipeline(stages=feature_pipeline.getStages() + [classifier])
    with mlflow.start_run(run_name=run_name):
        fitted = pipe.fit(fit_df)
        m = get_metrics(fitted.transform(val_df))
        mlflow.log_params({**params, "imbalance": "class_weight",
                           "fit_years": "2012-2014", "val_year": "2015"})
        mlflow.log_metrics({f"val_{k}": v for k, v in m.items()})
    print(run_name, {k: round(v, 4) for k, v in m.items()})
    return m

# Tune Logistic Regression: try 2 more regParam values
for reg in [0.01, 0.1]:
    lr = LogisticRegression(featuresCol="features", labelCol="label",
                            weightCol="class_weight", maxIter=50, regParam=reg)
    train_and_log(f"logreg_reg{reg}", lr,
                  {"algorithm": "LogisticRegression", "regParam": reg, "maxIter": 50})

Tuning regParam (0, 0.01, 0.1) barely changes results (AUC-ROC 0.710–0.711).
Logistic Regression is limited by its simple, straight-line patterns, not by overfitting. Next: a tree-based model.

## Step 8: Random Forest (50 trees), tuning maxDepth = 5, 8, 10

In [0]:
from pyspark.ml.classification import RandomForestClassifier

for depth in [5, 8, 10]:
    rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                                weightCol="class_weight", numTrees=50,
                                maxDepth=depth, seed=42)
    train_and_log(f"rf_depth{depth}", rf,
                  {"algorithm": "RandomForest", "numTrees": 50, "maxDepth": depth})

Random Forest improves with depth (AUC-ROC 0.699 → 0.704 → 0.706) but stays slightly below Logistic Regression (0.711).
A more complex model is not automatically better. All models reach ~0.70–0.71, suggesting the available features
(no FICO score) limit predictive power. **6 runs logged** (2 model types × 3 settings).

## Step 9: Gradient-Boosted Trees (50 rounds, depth 5), the optional 3rd model type
Each new tree corrects the mistakes of the previous ones.

In [0]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    weightCol="class_weight", maxIter=50, maxDepth=5, seed=42)
train_and_log("gbt_iter50_depth5", gbt,
              {"algorithm": "GBT", "maxIter": 50, "maxDepth": 5})

## Model choice
**Winner: GBT** (validation AUC-ROC 0.712, recall 60.4%, F1 0.493): the highest on all three.
Its AUC edge over Logistic Regression (0.711) is tiny, but its recall is ~4 points higher, meaning more defaults caught,
which is the most costly mistake for a lender. Logistic Regression remains a simpler, more explainable backup.
**7 runs logged** across 3 model types.

## Step 10: Final model: GBT retrained on all training years (2012–2015) and saved in MLflow

In [0]:
import mlflow.spark
from mlflow.models import infer_signature

final_gbt = GBTClassifier(featuresCol="features", labelCol="label",
                          weightCol="class_weight", maxIter=50, maxDepth=5, seed=42)
final_pipe = Pipeline(stages=feature_pipeline.getStages() + [final_gbt])

with mlflow.start_run(run_name="final_gbt") as run:
    final_model = final_pipe.fit(train)                          # learn on 2012-2015

    # Signature: which raw columns go in, what comes out
    sample = train.select(ALL_FEATURES).limit(5)
    signature = infer_signature(sample.toPandas(),
                                final_model.transform(sample).select("prediction").toPandas())

    mlflow.log_params({"algorithm": "GBT", "maxIter": 50, "maxDepth": 5,
                       "imbalance": "class_weight", "train_years": "2012-2015"})
    mlflow.spark.log_model(final_model, artifact_path="model", signature=signature,
                           dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/artifacts/mlflow_tmp")

print("Final run ID:", run.info.run_id)

Final GBT model trained on 2012–2015 (547,821 loans) and saved in MLflow with its input signature (run `final_gbt`).
Note: some inputs are integer columns; a production API should accept missing values as floats (see model card).

## Step 11: Final runner-up: Logistic Regression (regParam 0.1) retrained on 2012–2015 and saved
Needed so evaluation can compare both model types on the test set.

In [0]:
final_lr = LogisticRegression(featuresCol="features", labelCol="label",
                              weightCol="class_weight", maxIter=50, regParam=0.1)
final_lr_pipe = Pipeline(stages=feature_pipeline.getStages() + [final_lr])

with mlflow.start_run(run_name="final_logreg") as run_lr:
    final_lr_model = final_lr_pipe.fit(train)                    # learn on 2012-2015
    sample = train.select(ALL_FEATURES).limit(5)
    signature = infer_signature(sample.toPandas(),
                                final_lr_model.transform(sample).select("prediction").toPandas())
    mlflow.log_params({"algorithm": "LogisticRegression", "regParam": 0.1, "maxIter": 50,
                       "imbalance": "class_weight", "train_years": "2012-2015"})
    mlflow.spark.log_model(final_lr_model, artifact_path="model", signature=signature,
                           dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/artifacts/mlflow_tmp")

print("Final LR run ID:", run_lr.info.run_id)

Final Logistic Regression (regParam 0.1) trained on 2012–2015 and saved as run `final_logreg`, for comparison in evaluation.

# Training Summary
- **Imbalance strategy:** class weights (each default weighted 3.82×). No rows duplicated or removed.
- **Tuning:** time-based validation (fit 2012–2014, validate 2015), so the 2016 test set stays untouched.
- **7 MLflow runs, 3 model types:**
  | Model | Settings tried | Best val AUC-ROC | Recall |
  |---|---|---|---|
  | Logistic Regression | regParam 0, 0.01, 0.1 | 0.711 | 56.5% |
  | Random Forest (50 trees) | maxDepth 5, 8, 10 | 0.706 | 56.1% |
  | **GBT (50 rounds)** | maxDepth 5 | **0.712** | **60.4%** |
- **Winner:** GBT, retrained on 2012–2015 and saved in MLflow as run `final_gbt` (full pipeline: cleaning + model).
- **Runner-up:** Logistic Regression (regParam 0.1), also retrained and saved as run `final_logreg`.